Ячейка 1 — установка

In [ ]:
!pip install -q -r requirements.txt

Ячейка 2 — импорты

In [ ]:
import os
import json
import zipfile
import requests
import pandas as pd

from pipeline import run_pipeline, normalize_timestamp, UNIFIED_COLUMNS
from parsers.json_parser import parse_json_line
from parsers.syslog_parser import parse_syslog_5424
from parsers.evtx_parser import parse_evtx

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("OK")

Ячейка 3 — скачивание данных

In [ ]:
EVTX_URL = (
    "https://github.com/sbousseaden/EVTX-ATTACK-SAMPLES/raw/master/"
    "Execution/Exec_sysmon_meterpreter_reversetcp_msipackage.evtx"
)
if not os.path.exists("sample.evtx"):
    r = requests.get(EVTX_URL, timeout=60); r.raise_for_status()
    open("sample.evtx", "wb").write(r.content)
print("EVTX:", os.path.getsize("sample.evtx"), "байт")

CLOUDTRAIL_URL = (
    "https://media.githubusercontent.com/media/splunk/attack_data/master/"
    "datasets/attack_techniques/T1110.003/aws_multiple_login_fail_per_user/cloudtrail.json"
)
if not os.path.exists("cloudtrail.json"):
    r = requests.get(CLOUDTRAIL_URL, timeout=60); r.raise_for_status()
    open("cloudtrail.json", "w", encoding="utf-8").write(r.text)
print("CloudTrail:", os.path.getsize("cloudtrail.json"), "байт")

Ячейка 4 — прогон пайплайна на трёх источниках

In [ ]:
for path, kind in [
    ("cloudtrail.json", "json"),
    ("sample.evtx", "evtx"),
]:
    df = run_pipeline(path, kind)
    print(f"\n=== {kind} ===")
    print("shape:", df.shape)
    print(df.dtypes)
    display(df.head(3))

# Syslog — из демо-строк
sample_syslog = [
    "<34>1 2025-01-02T03:04:05.123Z host1 sshd 1234 ID47 [x] Failed password for alice",
    "<34>1 2025-01-02T03:04:06.200Z host1 sshd 1235 ID48 [x] Failed password for alice",
    "битая строка",
]
open("sample_syslog.log", "w", encoding="utf-8").write("\n".join(sample_syslog))
df_sys = run_pipeline("sample_syslog.log", "syslog")
print("\n=== syslog ===")
display(df_sys)

Ячейка 5 — контрольный пример + тесты

In [ ]:
# Контроль: 25 failed с 10.0.0.99 в demo_auth_log.csv (генерируется в лекции)
# Если демо-CSV нет — пропускаем; иначе:
if os.path.exists("demo_auth_log.csv"):
    check = pd.read_csv("demo_auth_log.csv", parse_dates=["timestamp"])
    n = len(check[(check["src_ip"] == "10.0.0.99") &
                  (check["event_type"] == "login_failed")])
    assert n == 25, f"ожидалось 25, получено {n}"
    print("Контрольный пример пройден ✓")

# Юнит-тесты
!pytest tests/ -v